# Setup Check
**Run this notebook before the workshop starts.**

Each cell prints `✓ PASS` or `✗ FAIL` with a fix hint.
You only need the cells marked *(required)* to participate in all exercises.
Docker / OpenSearch is *optional* — a local in memory fallback is provided.

There are known package dependency concerns. 

In recent years, the new Mac Silicon architecture, changes to setuptools and
pip, numpy v2.0, and changes to how Huggingface libraries are managed mean that
some of the latest versions of libraries are incompatible with older operating
systems. In order to meet the needs of attendees who may be participating on
different operating systems, I have made some adjustments to the required package
versions, often choosing to use an older version of a library, rather than the
latest version.

Friends have tested the versions specified in `pyproject.toml` to ensure that they work together on Windows, Mac Intel, Mac Silicon and Linux platforms.

However, it is possible that you may encounter package conflicts. Please contact me at least 48 hours in advance of the workshop, and I will do my best to find a solution before the workshop. I will also be available during the workshop to help troubleshoot any issues that arise, but may have limited time to help.

# Installation Instructions
Review the README.md and run installation commands in your terminal. When you are done, run this
notebook to check that all is working.  

**Run these commands in your terminal from the repo root:**
```
uv sync    
docker compose up -d
```

In [18]:
# ── Python version (required) ──────────────────────────────────────────────
import sys

v = sys.version_info
if v.major == 3 and v.minor == 12:
    print(f"✓ PASS  Python {v.major}.{v.minor}.{v.micro}")
else:
    print(f"✗ FAIL  Python {v.major}.{v.minor} — need 3.12. Install from python.org or use pyenv")

✓ PASS  Python 3.12.0


In [ ]:
# ── uv package manager (required) ─────────────────────────────────────────
import shutil

if shutil.which("uv") is None:
    print("✗ FAIL  uv is not installed.")
    print("        Install: https://docs.astral.sh/uv/getting-started/installation/")
else:
    print("✓ PASS  uv is installed")
    print("        Next step: run 'uv sync' from the repo root")

In [20]:
# ── Core packages (required) ───────────────────────────────────────────────
key_packages = [
    ("sentence_transformers", "sentence-transformers"),
    ("bertopic", "bertopic"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("emoji", "emoji"),
]
all_ok = True
for module, pkg in key_packages:
    try:
        __import__(module)
        print(f"✓ PASS  {pkg}")
    except ImportError:
        print(f"✗ FAIL  {pkg}  →  run: `uv sync` or `uv add {pkg}`")
        all_ok = False
if all_ok:
    print("\nAll key packages installed!")

✓ PASS  sentence-transformers
✓ PASS  bertopic
✓ PASS  numpy
✓ PASS  pandas
✓ PASS  emoji

All key packages installed!


In [14]:
# ── Files (required) ────────────────────────────────────────────────
from pathlib import Path

# Detect repo root by looking for a known marker file
repo_root = Path("..") if Path("../sample_posts.json").exists() else Path(".")

# Create output directory if it doesn't exist
output_dir = repo_root / "output"
output_dir.mkdir(exist_ok=True)
print(f"✓ PASS  output/  ({'already existed' if output_dir.exists() else 'created'})")

required_files = [
    "sample_posts.json",
]
for f in required_files:
    p = repo_root / f
    if p.exists():
        print(f"✓ PASS  {f}")
    else:
        print(f"✗ FAIL  {f}  →  make sure you cloned the full repo")

✓ PASS  output/  (already existed)
✓ PASS  sample_posts.json


In [15]:
# ── Embedding model (required — downloads ~90 MB on first run) ─────────────
from sentence_transformers import SentenceTransformer

try:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    emb = model.encode("hello world")
    assert emb.shape == (384,)
    print(f"✓ PASS  all-MiniLM-L6-v2  (dim={emb.shape[0]})")
except Exception as e:
    print(f"✗ FAIL  {e}")

✓ PASS  all-MiniLM-L6-v2  (dim=384)


In [16]:
# ── Elasticsearch (optional — Docker install) ────────────────────────────
import socket

try:
    s = socket.create_connection(("localhost", 9201), timeout=2)
    s.close()
    print("✓ PASS  Elasticsearch reachable on localhost:9201")
except OSError:
    print("○ SKIP  Elasticsearch not running — in-memory fallback will be used.")
    print("        To enable: docker compose up  (from repo root)")

✓ PASS  Elasticsearch reachable on localhost:9201


In [ ]:
# ── Ollama (optional — Docker install) ──────────────────────────────────────
import socket

try:
    s = socket.create_connection(("localhost", 11434), timeout=2)
    s.close()
    print("✓ PASS  Ollama reachable on localhost:11434")
except OSError:
    print("○ SKIP  Ollama not running — AI topic labelling will be skipped.")
    print("        To enable: docker compose up  (from repo root)")

In [17]:
# ── Streamlit (app.py demo) ─────────────────────────
try:
    import streamlit

    print(f"✓ PASS  streamlit {streamlit.__version__}")
except ImportError:
    print("○ SKIP  streamlit not installed — run: pip install streamlit")

✓ PASS  streamlit 1.56.0


## Run the test suite

Run the non-exercise tests to confirm your environment is wired up correctly. All tests should pass before you begin — the `exercise`-marked tests are intentionally stubbed out and will fail until you complete them.

```bash
uv run pytest -m "not exercise"
```

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-m", "not exercise", "--tb=short", "-q"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode == 0:
    print("✓ PASS  all non-exercise tests passed")
else:
    print("✗ FAIL  some tests failed — see output above")
    if result.stderr:
        print(result.stderr)

---
If all *required* cells show `✓ PASS`, you're ready for the workshop!

To launch the demo app:
```bash
uv run streamlit run app.py
```